# Projet B4 · Fiabiliser un assistant · ⭐⭐⭐

**Le problème** : « ça marche » n'est pas un résultat. Un assistant qui répond juste 6 fois sur 10 et invente le reste du temps a exactement l'air d'un assistant qui répond juste 10 fois sur 10 — il est tout aussi assuré dans les deux cas. Tant qu'on n'a pas mesuré, on ne sait rien.
**Ce qu'on construit** : un **banc de test** de 15 questions écrites à la main, une **mesure** reproductible (taux de bonnes réponses, taux d'hallucination, latence), trois **garde-fous** activables séparément, et le **avant / après** sur les mêmes 15 questions.
**Livrable** : ce notebook complété, le banc rangé en JSON, le tableau et le graphique avant / après, et la fiche projet finale avec ses quatre chiffres.

**Comment l'utiliser**
- Google Colab, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
- L'interrupteur `USE_MODEL` est en tête. **Tout ce projet se mesure en `USE_MODEL = False`** : le faux modèle est *déterministe*, deux exécutions donnent deux fois le même chiffre. C'est la condition pour comparer quoi que ce soit — avec un vrai modèle à température 0,7, un écart de 2 points ne voudrait rien dire. Repasse en `True` à la fin pour mesurer Qwen sur le même banc.
- Les cellules **« À toi »** sont des exercices : elles s'exécutent telles quelles, la vérification affiche ✅ ou ❌, la solution est cachée juste en dessous — essaie avant de l'ouvrir. Si tu sautes un exercice, le **filet de sécurité** replié juste après remet l'implémentation de référence et la suite fonctionne quand même.
- Ce projet mesure l'assistant du projet [B1](../B1-assistant-reviseur/). Les cellules de la section 1 en sont **recopiées telles quelles** : ce n'est pas le travail de ce projet, c'est son point de départ.

## 0. Préparation

La cellule `llm(messages)` de la séance 11, avec **une seule différence** : les branches du faux modèle suivent le protocole de ce projet. Quand on lui donne un contexte qui répond, il recopie le passage utile ; quand le contexte ne répond pas, **il invente** — sauf si la consigne système le lui interdit explicitement. C'est ce comportement qu'on va mesurer, puis corriger.

Lance ces trois cellules une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REFUS = "Je ne trouve pas cette information dans mes notes."

# mots trop fréquents pour compter comme du contenu ("le", "une", "quand"...)
STOP_FR = ("le la les l un une des du de d et ou à a au aux en dans sur par pour avec sans ce cet cette ces se son sa ses "
           "leur leurs il elle ils elles on ne pas plus que qui quoi quel quelle quels dont où quand comment est sont être avoir fait y t").split()
MOTS_OUTILS = {"quel", "quelle", "quels", "quelles", "comment", "combien", "pourquoi", "dans", "avec", "pour", "celui", "cette"}

def mots_utiles(texte):
    """Les mots « de contenu » d'un texte : au moins 4 lettres, hors mots vides et mots interrogatifs."""
    return {m for m in re.findall(r"\w{4,}", texte.lower())} - set(STOP_FR) - MOTS_OUTILS

print(len(STOP_FR), "mots vides ·", sorted(mots_utiles("Que capte la chlorophylle des chloroplastes ?")))

In [ ]:
# ---------- Mode démo : un faux LLM déterministe, sans réseau ni GPU ----------
def llm_factice(messages):
    """Avec un contexte qui répond, il recopie le passage utile ; sinon il INVENTE,
    sauf si la consigne système lui interdit explicitement de sortir du contexte."""
    question = messages[-1]["content"]
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "Contexte :" not in systeme:
        return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible."
    contexte = systeme.split("Contexte :", 1)[1]
    mots = mots_utiles(question)
    phrases = [p.strip(" -\n") for p in re.split(r"(?<=[.!?])\s+", contexte) if p.strip(" -\n")]
    meilleure = max(phrases, key=lambda p: sum(m in p.lower() for m in mots), default="")
    if sum(m in meilleure.lower() for m in mots) >= 2:      # le passage répond vraiment
        return "D'après tes documents : " + meilleure
    if "RÈGLE ABSOLUE" in systeme:                          # garde-fou 2 : la consigne renforcée
        return REFUS
    sujet = " ".join(sorted(mots, key=question.lower().find)[:3]) or "ce sujet"
    return (f"Bien sûr : {sujet}, c'est un mécanisme classique, décrit par Antoine Lavoisier "
            "en 1789 et rappelé au chapitre 4 du manuel.")

In [ ]:
# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice, déterministe)")

## 1. L'assistant à fiabiliser (recopié de B1)

Les cinq cellules qui suivent **viennent du projet B1**, à l'identique : le corpus de cours, le découpage en chunks, l'index TF-IDF avec ses mots vides français, la recherche `chercher_avec_seuil` et la fonction `repondre`. Ce ne sont **pas** des exercices : c'est l'assistant qu'on va mesurer, tel qu'il sort de la séance précédente. Si tu as fait B1 avec tes propres notes, colle-les dans `MES_NOTES` : tout le reste du notebook s'adaptera (mais il faudra réécrire le banc de test de la section 2, qui parle de SVT).

Une seule chose est ajoutée à la version B1 : `repondre_detail` porte **trois interrupteurs**, `gf_seuil`, `gf_consigne` et `gf_verif`, tous les trois **débranchés**. Ce sont les garde-fous de la section 5. Tant qu'ils sont à `False`, l'assistant se comporte exactement comme la version la plus naïve de B1 : il cherche les 3 meilleurs passages *quels que soient leurs scores* et laisse le modèle écrire. C'est volontaire — on veut mesurer le pire avant de le réparer.

In [ ]:
COURS_SVT = """La cellule est la plus petite unité du vivant. Tous les êtres vivants, du champignon à l'éléphant, sont formés d'une seule cellule ou de milliards de cellules. Une cellule animale contient un noyau, du cytoplasme et une membrane qui la sépare de l'extérieur.

La cellule végétale possède en plus trois éléments : une paroi rigide faite de cellulose, une grande vacuole remplie d'eau, et des chloroplastes qui contiennent la chlorophylle. Ces trois éléments manquent dans la cellule animale.

La photosynthèse est la fabrication de matière organique par les végétaux chlorophylliens. À la lumière, la plante utilise de l'eau et du dioxyde de carbone pour produire du glucose, et elle rejette du dioxygène.

La chlorophylle est le pigment vert des chloroplastes. Elle capte l'énergie lumineuse du Soleil et permet la réaction de photosynthèse. Sans lumière, la photosynthèse s'arrête complètement.

Les plantes prélèvent l'eau et les sels minéraux dans le sol par les poils absorbants des racines. Ce mélange, appelé sève brute, monte ensuite jusqu'aux feuilles par des vaisseaux conducteurs.

La respiration cellulaire concerne toutes les cellules vivantes, végétales comme animales. Elles consomment du dioxygène et rejettent du dioxyde de carbone, de jour comme de nuit, pour libérer l'énergie contenue dans le glucose.

Il ne faut pas confondre respiration et photosynthèse. La respiration a lieu en permanence dans tous les êtres vivants, alors que la photosynthèse n'a lieu qu'à la lumière et seulement chez les végétaux chlorophylliens.

Chez l'être humain, les échanges gazeux se font dans les poumons, au niveau de minuscules sacs appelés alvéoles pulmonaires. Leur surface totale est immense, ce qui rend le passage du dioxygène vers le sang très rapide.

Le dioxygène est transporté dans le sang par l'hémoglobine, une molécule contenue dans les globules rouges. C'est elle qui donne au sang sa couleur rouge.

La digestion transforme les aliments en nutriments assez petits pour passer dans le sang. Ce sont les enzymes digestives, présentes dans la salive, l'estomac et l'intestin, qui découpent les aliments.

L'absorption des nutriments se fait dans l'intestin grêle, dont la paroi est couverte de villosités. Ces replis multiplient la surface de contact entre les nutriments et le sang.

Les besoins alimentaires se répartissent en glucides, lipides, protides, vitamines, sels minéraux et eau. Les glucides et les lipides servent surtout d'énergie, les protides servent surtout à construire et à réparer.

L'appareil circulatoire distribue les nutriments et le dioxygène à toutes les cellules. Le cœur est une pompe qui fonctionne sans arrêt : au repos, il bat environ soixante-dix fois par minute chez un adulte.

Dans le sol, les décomposeurs, c'est-à-dire les bactéries, les champignons et les vers de terre, transforment la matière organique morte en matière minérale. Cette matière minérale redevient utilisable par les plantes.

Une chaîne alimentaire commence toujours par un producteur, un végétal chlorophyllien qui fabrique sa matière organique. Viennent ensuite les consommateurs qui le mangent, puis les décomposeurs. Toute l'énergie de la chaîne vient au départ du Soleil."""

MES_NOTES = """"""     # ← colle ton cours ici si tu as fait B1 avec tes notes

CORPUS = MES_NOTES.strip() if MES_NOTES.strip() else COURS_SVT
print("Corpus :", "TES notes" if MES_NOTES.strip() else "le cours d'exemple (SVT)", "·", len(CORPUS.split()), "mots")

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

print("outils prêts")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def decouper(texte):
    """Un paragraphe (séparé par une ligne vide) = un chunk."""
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

chunks = decouper(CORPUS)
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)

def vectoriser(textes):
    return tfidf.transform(textes).toarray()

vecteurs = vectoriser(chunks)
print(len(chunks), "chunks ·", vecteurs.shape[1], "mots de vocabulaire")

In [ ]:
SEUIL = 0.15      # le réglage de départ de B1 ; on le remettra en question à la section 5

def chercher(question, k=3):
    """Renvoie les k chunks les plus proches de la question, avec leur score."""
    scores = cosine_similarity(vectoriser([question]), vecteurs)[0]
    return [(chunks[i], round(float(scores[i]), 3)) for i in scores.argsort()[::-1][:k]]

def chercher_avec_seuil(question, k=3, seuil=SEUIL):
    """Comme chercher, mais on jette les passages trop peu proches de la question."""
    return [(p, s) for p, s in chercher(question, k) if s >= seuil]

print("dans le cours :", [s for _, s in chercher("Qu'est-ce qui transporte le dioxygène dans le sang ?")])
print("hors sujet    :", [s for _, s in chercher("Qui a écrit Les Misérables ?")])

In [ ]:
CONSIGNE_SIMPLE = ("Tu es l'assistant de révision de l'élève. Réponds en français, en 2 phrases maximum, "
                   "en t'appuyant sur le contexte ci-dessous.\n")

CONSIGNE_STRICTE = ("Tu es l'assistant de révision de l'élève. Réponds en français, en 2 phrases maximum.\n"
                    "RÈGLE ABSOLUE : réponds UNIQUEMENT avec ce qui est écrit dans le contexte ci-dessous.\n"
                    "Si la réponse n'y est pas, tu réponds exactement : " + REFUS + "\n"
                    "Ne devine jamais, n'ajoute aucune connaissance extérieure, n'invente aucun nom ni aucune date.\n")


def repondre_detail(question, k=3, seuil=SEUIL, gf_seuil=False, gf_consigne=False, gf_verif=False):
    """Le RAG de B1 + ses trois garde-fous, débranchés par défaut. Renvoie la réponse ET son contexte."""
    passages = chercher_avec_seuil(question, k, seuil) if gf_seuil else chercher(question, k)
    if not passages:                     # garde-fou 1 : on refuse AVANT même d'appeler le modèle
        return {"reponse": REFUS, "contexte": "", "score_max": 0.0, "appel_modele": False}
    contexte = "\n".join("- " + p for p, s in passages)
    systeme = (CONSIGNE_STRICTE if gf_consigne else CONSIGNE_SIMPLE) + "Contexte :\n" + contexte
    reponse = demander(question, systeme=systeme)
    if gf_verif and not est_un_refus(reponse) and not appuyee_par_le_contexte(reponse, contexte):
        reponse = REFUS                  # garde-fou 3 : vérification après coup
    return {"reponse": reponse, "contexte": contexte, "score_max": passages[0][1], "appel_modele": True}

def repondre(question, **options):
    """La fonction de B1 : on ne garde que le texte de la réponse."""
    return repondre_detail(question, **options)["reponse"]

In [ ]:
for q in ["Qu'est-ce qui transporte le dioxygène dans le sang ?",
          "Qui a écrit Les Misérables ?",
          "Quel est le prix d'un litre d'eau minérale au supermarché ?"]:
    sortie = repondre_detail(q)
    print(f"[meilleur score : {sortie['score_max']:.3f}] {q}")
    print(f"   → {sortie['reponse'][:150]}\n")

Regarde bien les deux dernières. L'assistant ne dit jamais qu'il ne sait pas : il répond du même ton assuré à une question de SVT et à une question sur Victor Hugo, en citant un savant et une date qui ne sont écrits nulle part. Et remarque la troisième : son meilleur score de recherche est **élevé** (le cours parle d'eau et de sels minéraux), donc un simple seuil ne la rattrapera pas.

C'est ça, une hallucination : pas un bug qui plante, une phrase bien formée et fausse. Le reste du notebook consiste à en compter combien il y en a, puis à les faire tomber.

## 2. Le banc de test

Un banc de test, c'est une liste de questions **écrites à la main, avant de voir les réponses de l'assistant**. Cet ordre n'est pas un détail : si tu écris tes questions après, tu écriras (sans le vouloir) celles auxquelles il répond déjà bien, et ton chiffre ne mesurera plus rien.

Quinze questions, en quatre familles :

| Famille | Combien | Ce qu'on teste | Comportement attendu |
|---|---|---|---|
| **dans le cours** | 8 | la réponse est noir sur blanc dans un paragraphe | répondre, avec le bon mot |
| **piège** | 3 | la même idée, posée avec **d'autres mots** que le cours | répondre quand même |
| **hors sujet** | 3 | aucune réponse possible dans le cours | **refuser** |
| **ambiguë** | 1 | plusieurs paragraphes répondent, en partie | répondre **ou** refuser : les deux se défendent |

Chaque ligne porte quatre choses : la `question`, la réponse `attendu`e en clair (pour un humain qui relit), les `mots_cles` obligatoires (pour la machine qui corrige) et le `comportement` attendu. Les huit premières lignes sont données en exemple ; les sept autres sont à toi.

In [ ]:
BANC_DANS_LE_COURS = [
    {"question": "Quel gaz la plante rejette-t-elle grâce à la photosynthèse ?", "attendu": "Du dioxygène.", "mots_cles": ["dioxygène"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Que contiennent les chloroplastes ?", "attendu": "De la chlorophylle.", "mots_cles": ["chlorophylle"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Qu'est-ce qui transporte le dioxygène dans le sang ?", "attendu": "L'hémoglobine des globules rouges.", "mots_cles": ["hémoglobine"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Où se font les échanges gazeux dans les poumons ?", "attendu": "Dans les alvéoles pulmonaires.", "mots_cles": ["alvéoles"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Où se fait l'absorption des nutriments ?", "attendu": "Dans l'intestin grêle.", "mots_cles": ["intestin"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "En quoi les décomposeurs transforment-ils la matière organique morte ?", "attendu": "En matière minérale.", "mots_cles": ["minérale"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Combien de fois le cœur bat-il par minute au repos ?", "attendu": "Environ soixante-dix fois.", "mots_cles": ["soixante-dix"], "comportement": "repondre", "famille": "dans le cours"},
    {"question": "Que produit la plante à partir de l'eau et du dioxyde de carbone ?", "attendu": "Du glucose.", "mots_cles": ["glucose"], "comportement": "repondre", "famille": "dans le cours"},
]
print(len(BANC_DANS_LE_COURS), "questions dont la réponse est dans les documents")

### À toi · exercice 1 ⭐⭐ · Écrire les 7 questions qui manquent

Complète `BANC_A_COMPLETER` avec **3 pièges**, **3 hors sujet** et **1 ambiguë**, au même format que les huit lignes ci-dessus. Le premier piège est écrit pour te donner le ton.

- un **piège** pose une question du cours avec d'autres mots que le cours (« pigment » au lieu de « chlorophylle », « nourriture » au lieu de « aliments ») : `comportement` = `"repondre"` ;
- un **hors sujet** n'a aucune réponse dans le cours : `mots_cles` = `[]`, `comportement` = `"refuser"`. Fais-en un qui **ressemble** au cours (des mots du cours, mais une question à laquelle il ne répond pas) : c'est le cas le plus difficile ;
- l'**ambiguë** a plusieurs réponses défendables : `comportement` = `"au choix"`.

Résultat attendu : `BANC` fait 15 lignes, réparties 8 / 3 / 3 / 1.

<details><summary>Indice</summary>

Recopie une ligne existante et change les valeurs. Les cinq clés doivent être présentes partout : `question`, `attendu`, `mots_cles`, `comportement`, `famille`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
BANC_A_COMPLETER = [
    {"question": "Quel pigment permet aux feuilles de capter la lumière du Soleil ?", "attendu": "La chlorophylle.", "mots_cles": ["chlorophylle"], "comportement": "repondre", "famille": "piège"},
    {"question": "Qu'est-ce qu'une cellule de plante a de plus qu'une cellule d'animal ?", "attendu": "Une paroi de cellulose, une vacuole, des chloroplastes.", "mots_cles": ["cellulose"], "comportement": "repondre", "famille": "piège"},
    {"question": "Comment la nourriture passe-t-elle dans le sang ?", "attendu": "La digestion la transforme en nutriments.", "mots_cles": ["nutriments"], "comportement": "repondre", "famille": "piège"},
    {"question": "Qui a écrit Les Misérables ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
    {"question": "Quelle est la capitale de l'Australie ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
    {"question": "Quel est le prix d'un litre d'eau minérale au supermarché ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
    {"question": "À quoi sert le glucose ?", "attendu": "À fournir de l'énergie aux cellules.", "mots_cles": ["énergie"], "comportement": "au choix", "famille": "ambiguë"},
]

BANC = BANC_DANS_LE_COURS + BANC_A_COMPLETER
print(len(BANC), "questions au banc")
```

</details>

In [ ]:
# À toi : ajoute les 6 lignes qui manquent après celle-ci (2 pièges, 3 hors sujet, 1 ambiguë)
BANC_A_COMPLETER = [
    {"question": "Quel pigment permet aux feuilles de capter la lumière du Soleil ?", "attendu": "La chlorophylle.", "mots_cles": ["chlorophylle"], "comportement": "repondre", "famille": "piège"},
]

BANC = BANC_DANS_LE_COURS + BANC_A_COMPLETER
print(len(BANC), "questions au banc")

In [ ]:
familles = ("dans le cours", "piège", "hors sujet", "ambiguë")
verifier("Exercice 1 · 15 questions en tout", lambda: len(BANC) == 15)
verifier("Exercice 1 · réparties 8 / 3 / 3 / 1", lambda: [sum(c["famille"] == f for c in BANC) for f in familles] == [8, 3, 3, 1])
verifier("Exercice 1 · les 5 champs partout", lambda: all(set(c) == {"question", "attendu", "mots_cles", "comportement", "famille"} for c in BANC))
verifier("Exercice 1 · les hors sujet attendent un refus", lambda: all(c["comportement"] == "refuser" and c["mots_cles"] == [] for c in BANC if c["famille"] == "hors sujet"))
verifier("Exercice 1 · les questions du cours ont des mots-clés", lambda: all(c["mots_cles"] for c in BANC if c["famille"] in ("dans le cours", "piège")))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Il réinjecte les 7 lignes de référence pour que la suite du notebook fonctionne quand même.
if len(BANC) != 15:
    BANC_A_COMPLETER = [
        {"question": "Quel pigment permet aux feuilles de capter la lumière du Soleil ?", "attendu": "La chlorophylle.", "mots_cles": ["chlorophylle"], "comportement": "repondre", "famille": "piège"},
        {"question": "Qu'est-ce qu'une cellule de plante a de plus qu'une cellule d'animal ?", "attendu": "Une paroi de cellulose, une vacuole, des chloroplastes.", "mots_cles": ["cellulose"], "comportement": "repondre", "famille": "piège"},
        {"question": "Comment la nourriture passe-t-elle dans le sang ?", "attendu": "La digestion la transforme en nutriments.", "mots_cles": ["nutriments"], "comportement": "repondre", "famille": "piège"},
        {"question": "Qui a écrit Les Misérables ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
        {"question": "Quelle est la capitale de l'Australie ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
        {"question": "Quel est le prix d'un litre d'eau minérale au supermarché ?", "attendu": REFUS, "mots_cles": [], "comportement": "refuser", "famille": "hors sujet"},
        {"question": "À quoi sert le glucose ?", "attendu": "À fournir de l'énergie aux cellules.", "mots_cles": ["énergie"], "comportement": "au choix", "famille": "ambiguë"},
    ]
    BANC = BANC_DANS_LE_COURS + BANC_A_COMPLETER
    print("filet : banc de référence remis en place —", len(BANC), "questions")

In [ ]:
banc = pd.DataFrame(BANC)

with open("banc_de_test.json", "w", encoding="utf-8") as f:      # réutilisable dans un autre notebook
    json.dump(BANC, f, ensure_ascii=False, indent=2)
print("banc_de_test.json écrit ·", len(banc), "questions")

banc["famille"].value_counts().to_frame("questions")

## 3. La mesure

Deux fonctions suffisent pour corriger automatiquement : une qui reconnaît un **refus**, une qui reconnaît une **bonne réponse**. Elles sont grossières toutes les deux, et c'est assumé — on préfère une mesure imparfaite mais *reproductible* à un jugement à l'œil qui change d'un jour à l'autre. On montrera à la fin de la section un cas où elles se trompent : le savoir fait partie de la mesure.

### À toi · exercice 2 ⭐ · Reconnaître un refus

Écris `est_un_refus(reponse)` : elle renvoie `True` si la réponse est un aveu d'ignorance. Cherche des formules, pas la phrase exacte — le modèle ne recopiera pas `REFUS` au caractère près.

Résultat attendu : `True` sur `REFUS` et sur « Désolé, je ne sais pas. », `False` sur « L'hémoglobine transporte le dioxygène. »

<details><summary>Indice</summary>

Une liste de marqueurs, et `any(m in reponse.lower() for m in MARQUEURS)`. Pense à « je ne trouve pas », « je ne sais pas », « n'est pas dans ».

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
MARQUEURS_DE_REFUS = ["je ne trouve pas", "je ne sais pas", "n'est pas dans", "pas cette information",
                      "aucune information", "je n'ai pas"]

def est_un_refus(reponse):
    """True si la réponse est un aveu d'ignorance plutôt qu'une affirmation."""
    bas = reponse.lower()
    return any(marqueur in bas for marqueur in MARQUEURS_DE_REFUS)

print(est_un_refus(REFUS), est_un_refus("Désolé, je ne sais pas."), est_un_refus("L'hémoglobine transporte le dioxygène."))
```

</details>

In [ ]:
# À toi
MARQUEURS_DE_REFUS = []

def est_un_refus(reponse):
    return False

print(est_un_refus(REFUS), est_un_refus("Désolé, je ne sais pas."), est_un_refus("L'hémoglobine transporte le dioxygène."))

In [ ]:
verifier("Exercice 2 · REFUS est bien un refus", lambda: est_un_refus(REFUS))
verifier("Exercice 2 · une autre formule marche aussi", lambda: est_un_refus("Désolé, je ne sais pas."))
verifier("Exercice 2 · une vraie réponse n'en est pas un", lambda: not est_un_refus("L'hémoglobine transporte le dioxygène."))
verifier("Exercice 2 · la casse n'a pas d'importance", lambda: est_un_refus(REFUS.upper()))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if not est_un_refus(REFUS):
    MARQUEURS_DE_REFUS = ["je ne trouve pas", "je ne sais pas", "n'est pas dans", "pas cette information",
                          "aucune information", "je n'ai pas"]
    def est_un_refus(reponse):
        return any(marqueur in reponse.lower() for marqueur in MARQUEURS_DE_REFUS)
    print("filet : est_un_refus remis en place")

### À toi · exercice 3 ⭐⭐ · Compter une bonne réponse

Écris `est_correcte(reponse, mots_cles)` : `True` si **tous** les mots-clés attendus apparaissent dans la réponse, sans tenir compte de la casse. Une liste de mots-clés vide renvoie `False` (on ne peut rien vérifier).

Résultat attendu : `True` pour `("Le sang contient de l'HÉMOGLOBINE.", ["hémoglobine"])`, `False` si le mot manque, `False` avec `[]`.

<details><summary>Indice</summary>

`all(mot.lower() in reponse.lower() for mot in mots_cles)`, et `bool(mots_cles)` devant pour le cas de la liste vide.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def est_correcte(reponse, mots_cles):
    """True si tous les mots-clés attendus apparaissent dans la réponse."""
    bas = reponse.lower()
    return bool(mots_cles) and all(mot.lower() in bas for mot in mots_cles)

print(est_correcte("Le sang contient de l'HÉMOGLOBINE.", ["hémoglobine"]),
      est_correcte("Le sang est rouge.", ["hémoglobine"]),
      est_correcte("Peu importe.", []))
```

</details>

In [ ]:
# À toi
def est_correcte(reponse, mots_cles):
    return False

print(est_correcte("Le sang contient de l'HÉMOGLOBINE.", ["hémoglobine"]),
      est_correcte("Le sang est rouge.", ["hémoglobine"]),
      est_correcte("Peu importe.", []))

In [ ]:
verifier("Exercice 3 · le mot-clé présent → correcte", lambda: est_correcte("Le sang contient de l'HÉMOGLOBINE.", ["hémoglobine"]))
verifier("Exercice 3 · le mot-clé absent → incorrecte", lambda: not est_correcte("Le sang est rouge.", ["hémoglobine"]))
verifier("Exercice 3 · sans mot-clé, on ne peut rien vérifier", lambda: not est_correcte("Peu importe.", []))
verifier("Exercice 3 · TOUS les mots-clés sont exigés", lambda: not est_correcte("De l'eau et du glucose.", ["glucose", "dioxygène"]))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if not est_correcte("Le sang contient de l'HÉMOGLOBINE.", ["hémoglobine"]):
    def est_correcte(reponse, mots_cles):
        bas = reponse.lower()
        return bool(mots_cles) and all(mot.lower() in bas for mot in mots_cles)
    print("filet : est_correcte remise en place")

### À toi · exercice 4 ⭐⭐⭐ · Passer tout le banc et chronométrer

Écris `evaluer(banc, **options)` : elle passe les 15 questions, mesure le temps de chacune avec `time.perf_counter()` et renvoie un **DataFrame** d'une ligne par question. Les `**options` sont passées telles quelles à `repondre_detail` (c'est ce qui permettra de comparer des réglages sans réécrire la fonction).

La règle de correction dépend du comportement attendu :

| `comportement` | la ligne est `ok` si… |
|---|---|
| `"repondre"` | ce n'est **pas** un refus **et** les mots-clés sont là |
| `"refuser"` | c'est un refus |
| `"au choix"` | c'est un refus **ou** les mots-clés sont là |

Colonnes attendues : `question`, `famille`, `ok`, `refus`, `score_max`, `appel_modele`, `ms`, `reponse`, `contexte`.

<details><summary>Indice</summary>

`depart = time.perf_counter()` avant l'appel, `(time.perf_counter() - depart) * 1000` après. Construis une liste de dictionnaires et finis par `pd.DataFrame(lignes)`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def evaluer(banc, **options):
    """Passe tout le banc et renvoie un DataFrame : une ligne par question."""
    lignes = []
    for cas in banc:
        depart = time.perf_counter()
        sortie = repondre_detail(cas["question"], **options)
        ms = (time.perf_counter() - depart) * 1000
        reponse = sortie["reponse"]
        refus = est_un_refus(reponse)
        if cas["comportement"] == "refuser":
            ok = refus
        elif cas["comportement"] == "au choix":
            ok = refus or est_correcte(reponse, cas["mots_cles"])
        else:
            ok = (not refus) and est_correcte(reponse, cas["mots_cles"])
        lignes.append({"question": cas["question"], "famille": cas["famille"], "ok": ok, "refus": refus,
                       "score_max": sortie["score_max"], "appel_modele": sortie["appel_modele"],
                       "ms": ms, "reponse": reponse, "contexte": sortie["contexte"]})
    return pd.DataFrame(lignes)

depart_du_projet = evaluer(BANC)
print(f"{depart_du_projet['ok'].mean():.0%} de bonnes réponses")
```

</details>

In [ ]:
# À toi
def evaluer(banc, **options):
    lignes = []
    for cas in banc:
        # chronomètre, appelle repondre_detail(cas["question"], **options), applique la règle du tableau
        pass
    return pd.DataFrame(lignes)

depart_du_projet = evaluer(BANC)
print(len(depart_du_projet), "lignes mesurées")

In [ ]:
attendues = {"question", "famille", "ok", "refus", "score_max", "appel_modele", "ms", "reponse", "contexte"}
verifier("Exercice 4 · une ligne par question", lambda: len(depart_du_projet) == len(BANC))
verifier("Exercice 4 · toutes les colonnes sont là", lambda: attendues <= set(depart_du_projet.columns))
verifier("Exercice 4 · le temps est mesuré et positif", lambda: (depart_du_projet["ms"] > 0).all())
verifier("Exercice 4 · les hors sujet sont comptés faux (aucun refus au départ)", lambda: not depart_du_projet[depart_du_projet["famille"] == "hors sujet"]["ok"].any())
verifier("Exercice 4 · la mesure est reproductible", lambda: evaluer(BANC)["ok"].tolist() == depart_du_projet["ok"].tolist())

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if len(depart_du_projet) != len(BANC):
    def evaluer(banc, **options):
        lignes = []
        for cas in banc:
            depart = time.perf_counter()
            sortie = repondre_detail(cas["question"], **options)
            ms = (time.perf_counter() - depart) * 1000
            reponse = sortie["reponse"]
            refus = est_un_refus(reponse)
            if cas["comportement"] == "refuser":
                ok = refus
            elif cas["comportement"] == "au choix":
                ok = refus or est_correcte(reponse, cas["mots_cles"])
            else:
                ok = (not refus) and est_correcte(reponse, cas["mots_cles"])
            lignes.append({"question": cas["question"], "famille": cas["famille"], "ok": ok, "refus": refus,
                           "score_max": sortie["score_max"], "appel_modele": sortie["appel_modele"],
                           "ms": ms, "reponse": reponse, "contexte": sortie["contexte"]})
        return pd.DataFrame(lignes)
    depart_du_projet = evaluer(BANC)
    print("filet : evaluer remise en place —", len(depart_du_projet), "lignes")

In [ ]:
print(f"=== Taux de bonnes réponses au départ : {depart_du_projet['ok'].mean():.0%} "
      f"({depart_du_projet['ok'].sum()}/{len(depart_du_projet)}) ===")
print(f"Latence : {depart_du_projet['ms'].mean():.2f} ms en moyenne · {depart_du_projet['ms'].median():.2f} ms en médiane")
print(f"Refus   : {depart_du_projet['refus'].sum()} sur {len(depart_du_projet)}\n")

depart_du_projet.groupby("famille")[["ok"]].mean().rename(columns={"ok": "part de bonnes réponses"})

In [ ]:
for _, ligne in depart_du_projet[~depart_du_projet["ok"]].iterrows():
    print(f"❌ [{ligne['famille']}] {ligne['question']}")
    print(f"     → {ligne['reponse'][:130]}\n")

**Le chiffre de départ : 60 % de bonnes réponses** — huit questions du cours sur huit, mais zéro refus sur les trois questions hors sujet. Autrement dit, l'assistant est bon quand la réponse existe et catastrophique quand elle n'existe pas, ce qui est exactement le profil dangereux : celui qu'on ne repère pas en l'essayant deux minutes sur des questions dont on connaît la réponse.

**Et les limites de la mesure, tout de suite.** Regarde la question du pigment. La réponse donnée est « *Elle capte l'énergie lumineuse du Soleil et permet la réaction de photosynthèse* » : c'est le bon passage, c'est la bonne information, et `est_correcte` la compte **fausse** parce que le mot « chlorophylle » a été remplacé par le pronom « Elle ». La mesure a tort et la réponse a raison. On garde le chiffre quand même — un thermomètre imparfait mais stable reste un thermomètre — mais on le sait, on le dit, et on y revient au banc de test final.

**Sur la latence** : c'est le seul chiffre de ce notebook qui n'est pas reproductible à l'identique, parce qu'il dépend de la machine. En mode démo il se compte en fractions de milliseconde et il est dominé par TF-IDF, pas par le « modèle ». Ce qui prédit vraiment la latence avec un vrai modèle, c'est le nombre d'**appels au modèle** — et ça, c'est un entier déterministe, qu'on suivra colonne `appel_modele`.

## 4. Les hallucinations

« Hallucination » est un mot qu'on emploie beaucoup et qu'on définit rarement. Ici il faut une définition **opérationnelle**, c'est-à-dire calculable par une fonction :

> Une hallucination est une **réponse affirmative** (donc pas un refus) **dont le contenu n'est pas appuyé par le contexte fourni** au modèle.

Deux précisions qui comptent. D'abord, ce n'est pas la même chose qu'une mauvaise réponse : une réponse peut être fausse tout en recopiant fidèlement un passage (l'assistant a lu le mauvais paragraphe) — c'est une erreur de recherche, pas une hallucination. Ensuite, « appuyé par le contexte » est une approximation : on regarde quelle **part des mots de contenu** de la réponse se retrouve dans le contexte. Au-dessus de la moitié, on considère que la réponse reste dans les clous.

### À toi · exercice 5 ⭐⭐⭐ · Mesurer l'appui d'une réponse sur son contexte

Écris `appuyee_par_le_contexte(reponse, contexte, part_min=0.5)` : elle renvoie `True` si au moins `part_min` des mots de contenu de la réponse (ceux que renvoie `mots_utiles`) apparaissent dans le contexte. Une réponse sans aucun mot de contenu renvoie `False`.

Résultat attendu : `True` pour une phrase recopiée du contexte, `False` pour une phrase qui cite Lavoisier et 1789 alors que le contexte parle de photosynthèse.

<details><summary>Indice</summary>

`mots = mots_utiles(reponse)` ; si vide → `False`. Sinon compte combien de ces mots sont dans `contexte.lower()` et divise par `len(mots)`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def appuyee_par_le_contexte(reponse, contexte, part_min=0.5):
    """Part des mots de contenu de la réponse qu'on retrouve dans le contexte fourni."""
    mots = mots_utiles(reponse)
    if not mots:
        return False
    bas = contexte.lower()
    return sum(1 for mot in mots if mot in bas) / len(mots) >= part_min

ctx = chunks[3]
print(appuyee_par_le_contexte("La chlorophylle est le pigment vert des chloroplastes.", ctx),
      appuyee_par_le_contexte("Décrit par Antoine Lavoisier en 1789 au chapitre 4 du manuel.", ctx))
```

</details>

In [ ]:
# À toi
def appuyee_par_le_contexte(reponse, contexte, part_min=0.5):
    return True

ctx = chunks[3]
print(appuyee_par_le_contexte("La chlorophylle est le pigment vert des chloroplastes.", ctx),
      appuyee_par_le_contexte("Décrit par Antoine Lavoisier en 1789 au chapitre 4 du manuel.", ctx))

In [ ]:
verifier("Exercice 5 · une phrase recopiée est appuyée", lambda: appuyee_par_le_contexte("La chlorophylle est le pigment vert des chloroplastes.", chunks[3]))
verifier("Exercice 5 · une phrase inventée ne l'est pas", lambda: not appuyee_par_le_contexte("Décrit par Antoine Lavoisier en 1789 au chapitre 4 du manuel.", chunks[3]))
verifier("Exercice 5 · une réponse vide n'est pas appuyée", lambda: not appuyee_par_le_contexte("", chunks[3]))
verifier("Exercice 5 · le seuil part_min est bien utilisé", lambda: appuyee_par_le_contexte("Décrit par Antoine Lavoisier.", chunks[3], part_min=0.0))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if appuyee_par_le_contexte("Décrit par Antoine Lavoisier en 1789 au chapitre 4 du manuel.", chunks[3]):
    def appuyee_par_le_contexte(reponse, contexte, part_min=0.5):
        mots = mots_utiles(reponse)
        if not mots:
            return False
        bas = contexte.lower()
        return sum(1 for mot in mots if mot in bas) / len(mots) >= part_min
    print("filet : appuyee_par_le_contexte remise en place")

In [ ]:
def est_hallucination(reponse, contexte):
    """Une affirmation (pas un refus) que le contexte fourni n'appuie pas."""
    if est_un_refus(reponse):
        return False
    return not appuyee_par_le_contexte(reponse, contexte)

def mesurer(banc, **options):
    """evaluer + la colonne hallucination : le tableau complet d'un réglage."""
    tableau = evaluer(banc, **options)
    tableau["hallucination"] = [est_hallucination(r, c) for r, c in zip(tableau["reponse"], tableau["contexte"])]
    return tableau

def resume(tableau):
    """Les chiffres d'un réglage, en un seul dictionnaire."""
    return {"bonnes": tableau["ok"].mean(), "hallucinations": tableau["hallucination"].mean(),
            "refus": tableau["refus"].mean(), "appels_modele": int(tableau["appel_modele"].sum()),
            "latence_moy_ms": tableau["ms"].mean(), "latence_med_ms": tableau["ms"].median()}

avant = mesurer(BANC)
print(f"=== Taux d'hallucination au départ : {avant['hallucination'].mean():.0%} "
      f"({avant['hallucination'].sum()}/{len(avant)}) ===")

In [ ]:
pires = avant[avant["hallucination"]].sort_values("score_max", ascending=False)
for rang, (_, ligne) in enumerate(pires.head(3).iterrows(), 1):
    print(f"{rang}. [{ligne['famille']} · meilleur score {ligne['score_max']:.3f}] {ligne['question']}")
    print(f"   → {ligne['reponse'][:140]}\n")

### À toi · exercice 6 ⭐⭐ · Classer les hallucinations par cause

Toutes les hallucinations ne se soignent pas pareil. Trois causes suffisent à les ranger :

- **passage manquant** : la recherche n'a rien ramené de pertinent (le meilleur score est sous le seuil) ;
- **question ambiguë** : la question elle-même ne désigne pas un passage précis ;
- **consigne trop faible** : la recherche a ramené des passages bien notés, mais ils ne répondent pas — et rien dans le prompt n'interdisait d'inventer.

Écris `cause_de_l_hallucination(ligne, seuil=SEUIL)` qui reçoit **une ligne** du tableau et renvoie une de ces trois chaînes, ou `"—"` si la ligne n'est pas une hallucination.

<details><summary>Indice</summary>

Teste dans cet ordre : pas d'hallucination → `"—"` ; famille `"ambiguë"` → `"question ambiguë"` ; `ligne["score_max"] < seuil` → `"passage manquant"` ; sinon `"consigne trop faible"`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def cause_de_l_hallucination(ligne, seuil=SEUIL):
    """Range une hallucination dans une des trois causes du cours."""
    if not ligne["hallucination"]:
        return "—"
    if ligne["famille"] == "ambiguë":
        return "question ambiguë"
    if ligne["score_max"] < seuil:
        return "passage manquant"
    return "consigne trop faible"

avant["cause"] = avant.apply(cause_de_l_hallucination, axis=1)
print(avant["cause"].value_counts().to_string())
```

</details>

In [ ]:
# À toi
def cause_de_l_hallucination(ligne, seuil=SEUIL):
    return "—"

avant["cause"] = avant.apply(cause_de_l_hallucination, axis=1)
print(avant["cause"].value_counts().to_string())

In [ ]:
verifier("Exercice 6 · les bonnes réponses n'ont pas de cause", lambda: (avant.loc[~avant["hallucination"], "cause"] == "—").all())
verifier("Exercice 6 · toutes les hallucinations sont classées", lambda: (avant.loc[avant["hallucination"], "cause"] != "—").all())
verifier("Exercice 6 · les trois causes sont représentées", lambda: set(avant["cause"]) - {"—"} == {"passage manquant", "question ambiguë", "consigne trop faible"})

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if set(avant["cause"]) == {"—"}:
    def cause_de_l_hallucination(ligne, seuil=SEUIL):
        if not ligne["hallucination"]:
            return "—"
        if ligne["famille"] == "ambiguë":
            return "question ambiguë"
        if ligne["score_max"] < seuil:
            return "passage manquant"
        return "consigne trop faible"
    avant["cause"] = avant.apply(cause_de_l_hallucination, axis=1)
    print("filet : cause_de_l_hallucination remise en place")

In [ ]:
avant.loc[avant["hallucination"], ["famille", "cause", "score_max", "question"]]

Les trois causes appellent trois réponses différentes, et c'est tout l'intérêt de les avoir séparées :

- **passage manquant** → un **seuil** de similarité règle le problème : sous le seuil, on refuse sans même appeler le modèle ;
- **consigne trop faible** → le seuil n'y peut rien (le score était de 0,357, bien au-dessus). Il faut **le dire au modèle** : la consigne renforcée ;
- **question ambiguë** → ni l'un ni l'autre ne suffit vraiment ; il reste la **vérification après coup**, qui remplace par un refus toute réponse qui ne s'appuie pas sur le contexte.

Un garde-fou par cause.

## 5. Les garde-fous

Les trois sont déjà câblés dans `repondre_detail`, chacun derrière son interrupteur, ce qui permet de mesurer **l'effet de chacun séparément** — c'est la seule façon de savoir lequel sert à quoi.

| | Interrupteur | Ce qu'il fait | Quand il agit |
|---|---|---|---|
| **1** | `gf_seuil` | ne garde que les passages dont le score atteint le seuil ; si plus rien ne passe, refuse | **avant** l'appel au modèle |
| **2** | `gf_consigne` | remplace `CONSIGNE_SIMPLE` par `CONSIGNE_STRICTE` (« RÈGLE ABSOLUE… ») | **pendant** |
| **3** | `gf_verif` | si la réponse ne reprend pas assez de mots du contexte, la remplace par un refus | **après** |

Le garde-fou 1 est le seul qui fasse gagner du temps : il évite complètement l'appel au modèle. Les deux autres en coûtent (le 3 ajoute un calcul, négligeable ici mais réel).

In [ ]:
REGLAGES = {
    "départ (rien)": {},
    "1 · seuil": dict(gf_seuil=True),
    "2 · consigne": dict(gf_consigne=True),
    "3 · vérification": dict(gf_verif=True),
    "les 3 ensemble": dict(gf_seuil=True, gf_consigne=True, gf_verif=True),
}

effets = pd.DataFrame({nom: resume(mesurer(BANC, **options)) for nom, options in REGLAGES.items()}).T
effets[["bonnes", "hallucinations", "refus"]] = effets[["bonnes", "hallucinations", "refus"]].map(lambda v: f"{v:.0%}")
effets

Ce tableau se lit ligne par ligne, et chaque ligne dit quelque chose de différent.

**Le seuil seul** monte les bonnes réponses de 60 à 73 % et divise les hallucinations par deux : il attrape « Les Misérables » et « la capitale de l'Australie », dont la recherche ne ramenait rien (score 0,000). Il **n'attrape pas** la question du prix de l'eau minérale, dont le score était de 0,357 — le cours parle d'eau et de sels minéraux, donc la recherche est contente. Un seuil ne sait pas si un passage *répond* à la question ; il sait seulement s'il lui *ressemble*.

**La consigne renforcée seule** et **la vérification après coup seule** font toutes les deux tomber les hallucinations à zéro et montent à 87 %. Elles agissent sur les mêmes cas mais pas au même endroit : la consigne empêche le modèle d'écrire l'invention, la vérification l'intercepte après coup. Sur un vrai modèle, la première serait moins fiable (un modèle désobéit) et la seconde resterait, elle, une garantie mécanique.

**Les trois ensemble** ne font pas mieux que le meilleur des trois — 87 %, zéro hallucination. Ce n'est pas décevant, c'est normal : ils se recouvrent largement. On les garde tous les trois quand même, parce qu'ils ne tombent pas en panne ensemble : le 1 protège du modèle bavard, le 2 protège du modèle rapide, le 3 protège du modèle qui a désobéi au 2.

**Une honnêteté à avoir** : le garde-fou 3 est exactement le détecteur d'hallucination de la section 4, branché en amont. Qu'il fasse tomber la mesure à zéro n'est donc pas une preuve — c'est presque une tautologie. Le seul chiffre non circulaire de la ligne « 3 · vérification », c'est le taux de bonnes réponses.

### À toi · exercice 7 ⭐⭐⭐ · Régler le seuil sur des données

`SEUIL = 0.15` vient de B1, où il avait été posé à la main. Il est temps de le mesurer.

Écris `balayer_seuils(banc, seuils)` : pour chaque valeur de seuil, elle mesure le banc **avec le garde-fou 1 seul** et renvoie un DataFrame d'une ligne par seuil, avec les colonnes de `resume` plus la colonne `seuil`.

Résultat attendu : 11 lignes, la part de refus qui ne redescend jamais quand le seuil monte, et les bonnes réponses qui plafonnent puis s'effondrent.

<details><summary>Indice</summary>

`for s in seuils:` → `t = mesurer(banc, gf_seuil=True, seuil=s)` → `lignes.append({"seuil": s, **resume(t)})`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def balayer_seuils(banc, seuils):
    """Une ligne par seuil : bonnes réponses, hallucinations, refus."""
    lignes = []
    for s in seuils:
        lignes.append({"seuil": s, **resume(mesurer(banc, gf_seuil=True, seuil=s))})
    return pd.DataFrame(lignes)

courbe = balayer_seuils(BANC, [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
print(courbe[["seuil", "bonnes", "hallucinations", "refus"]].to_string(index=False))
```

</details>

In [ ]:
# À toi
def balayer_seuils(banc, seuils):
    lignes = []
    return pd.DataFrame(lignes)

courbe = balayer_seuils(BANC, [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
print(len(courbe), "seuils balayés")

In [ ]:
verifier("Exercice 7 · 11 seuils balayés", lambda: len(courbe) == 11)
verifier("Exercice 7 · les colonnes attendues", lambda: {"seuil", "bonnes", "hallucinations", "refus"} <= set(courbe.columns))
verifier("Exercice 7 · les refus ne redescendent jamais", lambda: (courbe["refus"].diff().dropna() >= -1e-9).all())
verifier("Exercice 7 · le seuil le plus haut n'est pas le meilleur", lambda: courbe["bonnes"].idxmax() != len(courbe) - 1)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
if len(courbe) != 11:
    def balayer_seuils(banc, seuils):
        lignes = []
        for s in seuils:
            lignes.append({"seuil": s, **resume(mesurer(banc, gf_seuil=True, seuil=s))})
        return pd.DataFrame(lignes)
    courbe = balayer_seuils(BANC, [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
    print("filet : balayer_seuils remise en place —", len(courbe), "lignes")

In [ ]:
plt.figure(figsize=(7.5, 4))
plt.plot(courbe["seuil"], courbe["bonnes"], "o-", color="tab:green", label="bonnes réponses")
plt.plot(courbe["seuil"], courbe["refus"], "s-", color="tab:orange", label="refus")
plt.plot(courbe["seuil"], courbe["hallucinations"], "^-", color="tab:red", label="hallucinations")
plt.axvline(SEUIL, color="grey", linestyle="--", label=f"seuil retenu : {SEUIL}")
plt.xlabel("seuil de similarité")
plt.ylabel("part des 15 questions")
plt.ylim(-0.05, 1.05)
plt.title("Le seuil se règle sur des données, pas au hasard")
plt.legend()
plt.show()

In [ ]:
DEUX_REGLAGES = {
    "seuil 0,05": dict(gf_seuil=True, seuil=0.05),
    "seuil 0,15": dict(gf_seuil=True, seuil=0.15),
    "seuil 0,45": dict(gf_seuil=True, seuil=0.45),
    "seuil 0,15 + garde-fou 3": dict(gf_seuil=True, seuil=0.15, gf_verif=True),
}
comparatif = pd.DataFrame({nom: resume(mesurer(BANC, **o)) for nom, o in DEUX_REGLAGES.items()}).T
comparatif["bonnes (sur 15)"] = (comparatif["bonnes"] * len(BANC)).round().astype(int)
comparatif["hallucinations (sur 15)"] = (comparatif["hallucinations"] * len(BANC)).round().astype(int)
comparatif["refus (sur 15)"] = (comparatif["refus"] * len(BANC)).round().astype(int)
comparatif[["bonnes (sur 15)", "hallucinations (sur 15)", "refus (sur 15)", "appels_modele", "latence_med_ms"]]

In [ ]:
bas, haut = comparatif.loc["seuil 0,15"], comparatif.loc["seuil 0,45"]
perdues_seuil = int(bas["bonnes (sur 15)"] - haut["bonnes (sur 15)"])
evitees_seuil = int(bas["hallucinations (sur 15)"] - haut["hallucinations (sur 15)"])
print(f"Monter le seuil de 0,15 à 0,45 : {perdues_seuil} bonnes réponses perdues "
      f"pour {evitees_seuil} hallucinations évitées,")
print(f"et {int(haut['refus (sur 15)'])} refus sur {len(BANC)} au lieu de {int(bas['refus (sur 15)'])}.")
print(f"\nGarder 0,15 et ajouter le garde-fou 3 : "
      f"{int(comparatif.loc['seuil 0,15 + garde-fou 3', 'bonnes (sur 15)'])} bonnes réponses, "
      f"{int(comparatif.loc['seuil 0,15 + garde-fou 3', 'hallucinations (sur 15)'])} hallucination.")

### Ce que je recommande, et ce que ça coûte

**Premier constat, qui n'était pas prévu** : les seuils 0,05 et 0,15 donnent exactement les mêmes chiffres. Ce n'est pas un bug, c'est la nature de TF-IDF — soit un mot de la question est dans le passage et le score monte d'un coup au-dessus de 0,2, soit il n'y est pas et le score est nul. La zone entre 0,05 et 0,20 est vide sur ce banc. Conclusion utile : **choisir entre ces deux valeurs n'a aucun effet mesurable ici**, et prétendre le contraire serait de la fausse précision. C'est le genre de résultat qu'on n'obtient qu'en mesurant.

**La vraie décision se joue plus haut.** Pour attraper la dernière hallucination (le prix de l'eau minérale, score 0,357), il faut monter le seuil au-delà de 0,40 — et là, le prix se paie : **2 bonnes réponses perdues pour 2 hallucinations évitées**, et surtout un assistant qui refuse **10 questions sur 15**. Deux tiers de silence. Un assistant qui se tait ne se trompe jamais, et ne sert à rien : c'est le degré zéro de la fiabilité.

**Recommandation : garder le seuil à 0,15 et ajouter le garde-fou 3.** On obtient 13 bonnes réponses sur 15, zéro hallucination et 4 refus — sans perdre une seule bonne réponse. Le raisonnement tient en une phrase : *le seuil est un mauvais outil pour juger la pertinence, parce qu'il ne regarde que la question ; la vérification après coup regarde la réponse, c'est-à-dire la seule chose qui puisse être fausse.* Le seuil garde quand même son rôle, celui d'éviter un appel au modèle inutile sur les questions manifestement hors sujet — 15 appels au modèle deviennent 13, ce qui sur un vrai modèle est du temps et de l'argent.

**Et l'échange est-il bon ?** Ici, oui, et facilement : on n'échange rien du tout, on gagne sur les deux tableaux. Il le resterait même en perdant une ou deux bonnes réponses, pour une raison qui n'est pas mathématique : une hallucination sur un cours de révision, c'est une fausse information apprise par cœur avant un contrôle. Un refus, c'est cinq minutes perdues à relire le paragraphe soi-même. Les deux erreurs n'ont pas le même prix, donc les compter à égalité serait déjà une erreur de mesure.

## 6. Le avant / après

Mêmes 15 questions, même code, même corpus : seuls les trois interrupteurs changent. C'est la comparaison la plus honnête qu'on puisse faire — et elle n'a de sens que parce que tout est déterministe. En `USE_MODEL = False`, relancer ce notebook deux fois donne deux fois les mêmes ✅ et les mêmes ❌ ; c'est ce qui permet d'attribuer l'écart aux garde-fous et pas au hasard.

In [ ]:
apres = mesurer(BANC, gf_seuil=True, gf_consigne=True, gf_verif=True)

comparaison = pd.DataFrame({
    "famille": avant["famille"],
    "question": avant["question"].str.slice(0, 52),
    "avant": np.where(avant["ok"], "✅", "❌"),
    "hallu. avant": np.where(avant["hallucination"], "⚠️", ""),
    "après": np.where(apres["ok"], "✅", "❌"),
    "hallu. après": np.where(apres["hallucination"], "⚠️", ""),
})
comparaison

In [ ]:
mesures = ["bonnes", "hallucinations", "refus"]
valeurs_avant = [resume(avant)[m] for m in mesures]
valeurs_apres = [resume(apres)[m] for m in mesures]
x = np.arange(len(mesures))

plt.figure(figsize=(7.5, 4))
plt.bar(x - 0.2, valeurs_avant, 0.4, label="avant (aucun garde-fou)", color="tab:red")
plt.bar(x + 0.2, valeurs_apres, 0.4, label="après (les 3 garde-fous)", color="tab:green")
for i, (u, v) in enumerate(zip(valeurs_avant, valeurs_apres)):
    plt.text(i - 0.2, u + 0.02, f"{u:.0%}", ha="center")
    plt.text(i + 0.2, v + 0.02, f"{v:.0%}", ha="center")
plt.xticks(x, ["bonnes réponses", "hallucinations", "refus"])
plt.ylim(0, 1.15)
plt.ylabel("part des 15 questions")
plt.title("Avant / après, sur le même banc de 15 questions")
plt.legend()
plt.show()

In [ ]:
perdues = [q for q, a, p in zip(avant["question"], avant["ok"], apres["ok"]) if a and not p]
gagnees = [q for q, a, p in zip(avant["question"], avant["ok"], apres["ok"]) if p and not a]
resistent = [q for q, a, p in zip(avant["question"], avant["ok"], apres["ok"]) if not a and not p]

print(f"Bonnes réponses gagnées : {len(gagnees)}")
print(f"Bonnes réponses perdues : {len(perdues)}   ← le coût des garde-fous")
print(f"Hallucinations évitées  : {int(avant['hallucination'].sum() - apres['hallucination'].sum())}")
print(f"Appels au modèle        : {int(avant['appel_modele'].sum())} → {int(apres['appel_modele'].sum())} sur {len(BANC)} questions")
print(f"Latence médiane         : {avant['ms'].median():.2f} ms → {apres['ms'].median():.2f} ms")
print("\nQuestions qui résistent encore :")
for q in resistent:
    print("  ·", q)

**60 % → 87 % de bonnes réponses, 27 % → 0 % d'hallucinations, pour zéro bonne réponse perdue.** L'échange était gratuit ici, et il faut dire pourquoi plutôt que de s'en féliciter : parce que le seuil retenu (0,15) est en dessous du score de *toutes* les questions légitimes du banc, la plus basse étant à 0,250. Sur un banc plus dur — des questions plus mal formulées, un corpus plus gros où les scores se tassent — cette marge disparaîtrait et le coût réapparaîtrait, exactement comme la courbe de la section 5 le montre au-delà de 0,40.

Le prix payé est ailleurs, et il est réel : l'assistant refuse maintenant **4 questions sur 15** au lieu de zéro. Trois de ces refus sont mérités (les hors sujet). Le quatrième est la question ambiguë « À quoi sert le glucose ? », dont la réponse *est* dans le cours, en morceaux, répartie sur deux paragraphes. On l'a comptée juste (`"au choix"`), mais un élève qui pose cette question repart les mains vides. C'est le vrai coût, et il ne se voit pas dans le taux de bonnes réponses.

Sur la **latence**, le résultat est franc : elle ne bouge pas (fractions de milliseconde, dominées par TF-IDF). Le chiffre à retenir n'est pas celui-là mais les **15 → 13 appels au modèle** : avec Qwen sur un CPU Colab, chaque appel coûte plusieurs secondes, et deux appels évités sur quinze est la seule économie que ce notebook puisse promettre honnêtement.

### Le banc de test du notebook

Sept vérifications sur le travail lui-même. **La dernière échoue exprès** : lis l'explication juste après.

In [ ]:
verifier("Le banc fait 15 questions en 4 familles", lambda: len(BANC) == 15 and banc["famille"].nunique() == 4)
verifier("La mesure est reproductible (deux passages, mêmes ✅)", lambda: mesurer(BANC)["ok"].tolist() == avant["ok"].tolist())
verifier("Le taux de bonnes réponses a monté", lambda: apres["ok"].mean() > avant["ok"].mean())
verifier("Les hallucinations sont tombées à zéro", lambda: apres["hallucination"].sum() == 0)
verifier("Les 3 questions hors sujet reçoivent un refus", lambda: apres.loc[apres["famille"] == "hors sujet", "refus"].all())
verifier("Aucune bonne réponse n'a été perdue", lambda: len(perdues) == 0)

pigment = apres[apres["question"].str.contains("pigment")]
verifier("ÉCHEC VOLONTAIRE · la question « pigment » est comptée juste",
         lambda: len(pigment) > 0 and bool(pigment["ok"].iloc[0]))
print("\nCe que l'assistant a répondu :", pigment["reponse"].iloc[0] if len(pigment) else "(question absente du banc)")

**L'échec volontaire, et pourquoi il reste.** La question « Quel pigment permet aux feuilles de capter la lumière du Soleil ? » reçoit la réponse : « *Elle capte l'énergie lumineuse du Soleil et permet la réaction de photosynthèse.* » C'est le bon paragraphe, c'est la bonne information, un correcteur humain mettrait le point. `est_correcte` met zéro, parce qu'elle cherche le mot « chlorophylle » et que le passage dit « Elle ».

Ce ❌ ne mesure donc pas l'assistant : il mesure **la mesure**. Et c'est pour ça qu'on le laisse afficher un ❌ au lieu de le corriger discrètement. Trois façons de le lire :

1. **Le vrai taux est meilleur que 87 %.** Sur ces 15 questions, `est_correcte` produit au moins un faux négatif. Un taux mesuré est un plancher, jamais une note exacte.
2. **On pourrait tricher.** Il suffirait d'ajouter « pigment » ou « capte » aux `mots_cles` de cette ligne pour passer à 93 %. Ce serait ajuster le banc aux réponses obtenues — précisément ce que la section 2 interdisait. Le banc a été écrit avant ; il ne bouge plus.
3. **La correction propre est ailleurs.** Il faudrait résoudre le pronom (« Elle » → « La chlorophylle », phrase précédente), ou faire juger la réponse par un second modèle (« LLM juge », voir *Pour aller plus loin*), et vérifier que son verdict et le tien coïncident. Tant que ce n'est pas fait, la limite se déclare, elle ne se cache pas.

L'autre question qui résiste — « Qu'est-ce qu'une cellule de plante a de plus qu'une cellule d'animal ? » — est un échec d'une autre nature, celui-là bien réel : la recherche ramène le paragraphe sur la cellule *animale*, parce que TF-IDF compte les mots et ne voit pas le « de plus que ». Aucun garde-fou ne la répare : ils empêchent d'inventer, pas de chercher au mauvais endroit.

## 7. Conclusion et fiche projet

À retenir :

- **Un chiffre qu'on ne peut pas reproduire n'est pas une mesure.** Tout ce notebook tient sur le déterminisme du mode démo : mêmes questions, même code, mêmes résultats. C'est ce qui permet d'attribuer un écart aux garde-fous plutôt qu'au hasard du tirage.
- **Le banc s'écrit avant.** Sinon on écrit les questions auxquelles l'assistant sait déjà répondre, et le taux ne mesure plus que notre indulgence.
- **« Hallucination » doit être une fonction, pas une impression.** Ici : réponse affirmative non appuyée par le contexte fourni. Discutable, approximative — mais calculable, donc comparable d'une version à l'autre.
- **Un seuil juge la question, une vérification juge la réponse.** C'est pour ça que le seuil ne rattrape pas la question sur le prix de l'eau minérale, et que la vérification après coup, oui.
- **Tout garde-fou a un prix.** Ici il était nul en bonnes réponses, mais réel en refus : quatre questions sur quinze restent sans réponse. Un bilan honnête donne les deux chiffres.

Remplis la fiche ci-dessous : c'est ce que tu montreras.

In [ ]:
MA_SYNTHESE = """(à remplir) Mon assistant part de ... % de bonnes réponses et ... % d'hallucinations.
Les trois garde-fous le montent à ... % et ... %, au prix de ... refus sur 15.
La question qui résiste est ... , parce que ... .
Pour la régler, je testerais ... ."""

print("=== FICHE PROJET B4 · FIABILISER UN ASSISTANT ===")
print(f"Banc de test        : {len(BANC)} questions · " + " · ".join(f"{n} {f}" for f, n in banc['famille'].value_counts().items()))
print(f"Bonnes réponses     : {avant['ok'].mean():.0%}  →  {apres['ok'].mean():.0%}")
print(f"Hallucinations      : {avant['hallucination'].mean():.0%}  →  {apres['hallucination'].mean():.0%}")
print(f"Refus               : {avant['refus'].mean():.0%}  →  {apres['refus'].mean():.0%}")
print(f"Latence médiane     : {avant['ms'].median():.2f} ms  →  {apres['ms'].median():.2f} ms  "
      f"({int(avant['appel_modele'].sum())} → {int(apres['appel_modele'].sum())} appels au modèle)")
print(f"Coût des garde-fous : {len(perdues)} bonne(s) réponse(s) perdue(s) pour "
      f"{int(avant['hallucination'].sum() - apres['hallucination'].sum())} hallucination(s) évitée(s)")
print(f"Seuil retenu        : {SEUIL} (réglé sur la courbe de la section 5)")
print(f"Faiblesse qui reste : {resistent[0] if resistent else '(aucune)'}")
print(f"Mode                : {'modèle Qwen2.5-0.5B' if USE_MODEL else 'démo (llm_factice, déterministe)'}")
print("\nMa synthèse :", MA_SYNTHESE)

## Pour aller plus loin

- **Repasse le banc en `USE_MODEL = True`.** Les 15 questions ne changent pas, le code non plus : seul le modèle change. Lance-le **trois fois** et note les trois taux — l'écart entre eux est la marge d'erreur de toute mesure faite sur un modèle à température non nulle. C'est l'argument le plus convaincant du projet, et il tient en trois chiffres.
- **Fais juger les réponses par un second modèle** (« LLM juge ») au lieu des mots-clés, puis compare son verdict au tien sur les 15 questions : sur combien êtes-vous d'accord ? Commence par la question du pigment, celle où `est_correcte` se trompe. Recettes : https://huggingface.co/learn/cookbook/en/llm_judge
- **Ajoute la citation obligatoire** : l'assistant doit indiquer le numéro du passage utilisé (`[3]`), et une fonction vérifie que ce numéro existe et que le passage contient bien la réponse. Un quatrième garde-fou, et le plus vérifiable de tous.
- **Élargis le banc à 50 questions** en gardant les mêmes proportions. Les taux deviennent plus stables : sur 15 questions, une seule ligne pèse 7 points.
- **Regarde comment les bibliothèques d'évaluation formalisent tout ça** (fidélité, pertinence du contexte) : https://docs.ragas.io/en/stable/ — les métriques sont les tiennes, en plus habillées. Le code : https://github.com/explodinggradients/ragas
- Le modèle utilisé : https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct · un pipeline RAG de référence : https://python.langchain.com/docs/tutorials/rag/ · ce qu'est un agent et pourquoi l'évaluer est le vrai travail : https://www.kaggle.com/whitepaper-agents